In [21]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer

# 1. Cargar los mismos fragmentos y embeddings ya generados por el equipo
carpeta = Path("../data/processed")  # Usa Path("data/processed") si lo corres desde la raíz
df = pd.concat(
    [pd.read_csv(f) for f in sorted(carpeta.glob("chunks_*.csv"))],
    ignore_index=True,
)
vectores = np.load(carpeta / "embeddings.npy")

assert len(df) == len(vectores), "No coinciden los fragmentos con los embeddings"
print(f"Fragmentos cargados: {len(df)} | Dimensión vectores: {vectores.shape}")

Fragmentos cargados: 89 | Dimensión vectores: (89, 384)


In [22]:
# 2. Conectar al contenedor de ChromaDB (puerto 8000)
cliente = chromadb.PersistentClient(path="./chroma_local")

# Reiniciar la colección si ya existía para tener una prueba limpia
nombre_coleccion = "documentos_cnbs"
try:
    cliente.delete_collection(name=nombre_coleccion)
except Exception:
    pass

# Crear colección configurada explícitamente con Similitud del Coseno (Índice HNSW interno)
coleccion = cliente.create_collection(
    name=nombre_coleccion,
    metadata={
        "hnsw:space": "cosine",   # Métrica obligatoria según Actividad 5
        "hnsw:M": 16,
        "hnsw:construction_ef": 64,
    },
)

In [17]:
# 3. Preparar datos e insertar en ChromaDB (IDs, embeddings, documentos y metadatos)
ids = [f"doc_{i}" for i in range(len(df))]
documentos = df["contenido"].tolist()
metadatos = [
    {
        "documento": str(fila.documento),
        "categoria": str(fila.categoria),
        "fuente": str(fila.fuente),
        "seccion": str(fila.seccion),
        "numero_item": int(fila.numero_item),
    }
    for fila in df.itertuples()
]

coleccion.add(
    ids=ids,
    embeddings=vectores.tolist(),
    documents=documentos,
    metadatas=metadatos,
)
print("Registros almacenados en ChromaDB:", coleccion.count())

Registros almacenados en ChromaDB: 20


In [18]:
# 4. Consulta de prueba semántica (equivalente a buscar_pgvector)
modelo = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

def buscar_chromadb(pregunta, k=3):
    v = modelo.encode(pregunta).tolist()
    res = coleccion.query(
        query_embeddings=[v],
        n_results=k,
        include=["documents", "distances", "metadatas"]
    )
    # ChromaDB devuelve distancia coseno (0 = idéntico); similitud = 1 - distancia
    resultados = []
    for doc, dist in zip(res["documents"][0], res["distances"][0]):
        resultados.append((doc, 1 - dist))
    return resultados

print("\n--- Prueba de consulta en ChromaDB ---")
for texto, score in buscar_chromadb("puedo consultar gratis mi historial crediticio", k=3):
    print(f"Similitud: {score:.4f}\n{texto}\n")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1881.15it/s]



--- Prueba de consulta en ChromaDB ---
Similitud: 0.4339
Obligaciones de las instituciones supervisadas por la CNBS: Permitir al usuario la consulta gratuita y por los canales que la institución supervisada establezca, en relación a: Estado de sus cuentas, créditos, servicios y productos, incluyendo la información que la institución supervisada reporta a la Central de Información Crediticia (CIC).

Similitud: 0.3872
Deberes y buenas prácticas del usuario financiero: Evitar sobre endeudarse con las instituciones supervisadas y con particulares, velando por la economía y bienestar financiero del hogar.

Similitud: 0.3832
Obligaciones de las instituciones supervisadas por la CNBS: No requerir nuevamente al usuario financiero, información que previamente fue entregada a la institución supervisada, sin perjuicio de la información del usuario financiero que deba mantenerse actualizada.



In [20]:
preguntas = [
    "puedo consultar gratis mi historial crediticio",
    "que debe hacer el usuario antes de firmar un contrato",
    "la institucion debe atender mis reclamos",
    "como evitar el sobreendeudamiento",
    "que informacion debe darme el banco sobre mis cuentas",
    "quien supervisa a las instituciones financieras",
    "puedo pedir una copia de mi contrato",
    "como proteger mis datos personales",
    "que hago si cobran comisiones no informadas",
    "derechos del usuario financiero",
]

vectores_preg = {p: modelo.encode(p).tolist() for p in preguntas}

def consulta_chromadb(v, k):
    return coleccion.query(query_embeddings=[v], n_results=k, include=[])

# Calentamiento (3 consultas sin medir)
for v in list(vectores_preg.values())[:3]:
    consulta_chromadb(v, 3)

filas_tiempos = []
for k in (3, 5, 10):
    for p, v in vectores_preg.items():
        t0 = time.perf_counter()
        consulta_chromadb(v, k)
        ms = (time.perf_counter() - t0) * 1000
        filas_tiempos.append({"base": "chromadb", "k": k, "pregunta": p, "ms": ms})

tiempos = pd.DataFrame(filas_tiempos)
tiempos.to_csv(carpeta / "tiempos_chromadb.csv", index=False)

print("--- Resumen de tiempos ChromaDB (ms) ---")
print(tiempos.groupby("k")["ms"].agg(["min", "max", "mean"]).round(3))

--- Resumen de tiempos ChromaDB (ms) ---
      min    max   mean
k                      
3   0.832  1.108  0.986
5   0.808  1.126  0.973
10  0.815  0.952  0.891
